**Ambil data dari Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

raw_path = "/content/drive/MyDrive/Tugas Big-Data/OULAD_EarlyWarning/data/raw/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

courses = pd.read_csv(raw_path + "courses.csv")
assessments = pd.read_csv(raw_path + "assessments.csv")
vle = pd.read_csv(raw_path + "vle.csv")
studentInfo = pd.read_csv(raw_path + "studentInfo.csv")
studentRegistration = pd.read_csv(raw_path + "studentRegistration.csv")
studentAssessment = pd.read_csv(raw_path + "studentAssessment.csv")
studentVle = pd.read_csv(raw_path + "studentVle.csv")

In [ ]:
print(courses.shape)
print(assessments.shape)
print(vle.shape)
print(studentInfo.shape)
print(studentRegistration.shape)
print(studentAssessment.shape)
print(studentVle.shape)

(22, 3)
(206, 6)
(6364, 6)
(32593, 12)
(32593, 5)
(173912, 5)
(10655280, 6)


**DATA PROCESSING - CLEANING, TRANSFORMASI, INTEGRASI**

In [ ]:
import numpy as np

# Salin dataframe agar data asli tetap aman
courses_clean = courses.copy()
assessments_clean = assessments.copy()
vle_clean = vle.copy()
studentInfo_clean = studentInfo.copy()
studentRegistration_clean = studentRegistration.copy()
studentAssessment_clean = studentAssessment.copy()
studentVle_clean = studentVle.copy()


**1. CLEANING**

In [ ]:
# assessments: diisi missing pada date dengan median
assessments_clean["date"] = assessments_clean["date"].fillna(assessments_clean["date"].median())

# vle: diisi missing week_from dan week_to dengan 0
vle_clean["week_from"] = vle_clean["week_from"].fillna(0)
vle_clean["week_to"] = vle_clean["week_to"].fillna(0)

# studentInfo: diisi missing imd_band dengan Unknown
studentInfo_clean["imd_band"] = studentInfo_clean["imd_band"].fillna("Unknown")

# studentRegistration: diisi missing date_registration dengan median
studentRegistration_clean["date_registration"] = studentRegistration_clean["date_registration"].fillna(
    studentRegistration_clean["date_registration"].median()
)

# studentAssessment: diisi missing score dengan 0
studentAssessment_clean["score"] = studentAssessment_clean["score"].fillna(0)


# cek missing values untuk memastikan cleaning berhasil
# cek missing values setelah cleaning
for name, df in {
    "assessments_clean": assessments_clean,
    "vle_clean": vle_clean,
    "studentInfo_clean": studentInfo_clean,
    "studentRegistration_clean": studentRegistration_clean,
    "studentAssessment_clean": studentAssessment_clean
}.items():
    print(f"\n{name}")
    print(df.isnull().sum())


assessments_clean
code_module          0
code_presentation    0
id_assessment        0
assessment_type      0
date                 0
weight               0
dtype: int64

vle_clean
id_site              0
code_module          0
code_presentation    0
activity_type        0
week_from            0
week_to              0
dtype: int64

studentInfo_clean
code_module             0
code_presentation       0
id_student              0
gender                  0
region                  0
highest_education       0
imd_band                0
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
final_result            0
dtype: int64

studentRegistration_clean
code_module                0
code_presentation          0
id_student                 0
date_registration          0
date_unregistration    22521
dtype: int64

studentAssessment_clean
id_assessment     0
id_student        0
date_submitted    0
is_banked         0
score             0
dtype: int64


**2. Transformasi**

In [ ]:
# Buat flag withdrawal
studentRegistration_clean["is_withdrawn"] = studentRegistration_clean["date_unregistration"].notna().astype(int)

# Buat flag submit assessment
studentAssessment_clean["submitted"] = (studentAssessment_clean["date_submitted"] >= 0).astype(int)

# Total klik per mahasiswa per modul
studentVle_agg = (
    studentVle_clean
    .groupby(["id_student", "code_module", "code_presentation"], as_index=False)
    .agg(
        total_clicks=("sum_click", "sum"),
        avg_clicks=("sum_click", "mean"),
        last_interaction_day=("date", "max"),
        active_days=("date", "nunique")
    )
)

# Ringkasan asesmen per mahasiswa per modul
studentAssessment_merged = pd.merge(
    studentAssessment_clean,
    assessments_clean[["id_assessment", "code_module", "code_presentation", "assessment_type", "weight"]],
    on="id_assessment",
    how="left"
)

studentAssessment_agg = (
    studentAssessment_merged
    .groupby(["id_student", "code_module", "code_presentation"], as_index=False)
    .agg(
        avg_score=("score", "mean"),
        max_score=("score", "max"),
        total_assessments=("id_assessment", "count"),
        total_submitted=("submitted", "sum")
    )
)

# Buat assessment completion rate
studentAssessment_agg["assessment_completion_rate"] = (
    studentAssessment_agg["total_submitted"] / studentAssessment_agg["total_assessments"]
)


**3. INTEGRASI**

In [ ]:
# Gabungkan studentInfo + registration
merged_data = pd.merge(
    studentInfo_clean,
    studentRegistration_clean,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

# Gabungkan hasil agregasi VLE
merged_data = pd.merge(
    merged_data,
    studentVle_agg,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

# Gabungkan hasil agregasi assessment
merged_data = pd.merge(
    merged_data,
    studentAssessment_agg,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

# Gabungkan lama modul dari courses
merged_data = pd.merge(
    merged_data,
    courses_clean,
    on=["code_module", "code_presentation"],
    how="left"
)


**MENANGANI MISSING SETELAH MERGE**

In [ ]:
# Isi missing numerik hasil agregasi dengan 0
from IPython.display import display, Markdown

num_cols = [
    "total_clicks", "avg_clicks", "last_interaction_day", "active_days",
    "avg_score", "max_score", "total_assessments", "total_submitted",
    "assessment_completion_rate", "length"
]

for col in num_cols:
    if col in merged_data.columns:
        merged_data[col] = merged_data[col].fillna(0)

# Cek hasil tranformasi dan integrasi
transformasi_summary = pd.DataFrame([
    ["studentVle_agg", studentVle.shape[0], studentVle_agg.shape[0], ", ".join(studentVle_agg.columns)],
    ["studentAssessment_agg", studentAssessment.shape[0], studentAssessment_agg.shape[0], ", ".join(studentAssessment_agg.columns)]
], columns=["Dataset Hasil", "Baris Sebelum", "Baris Sesudah", "Kolom/Fitur Baru"])

display(transformasi_summary)


display(Markdown("### Preview Hasil Transformasi - studentVle_agg"))
display(studentVle_agg.head(10))

display(Markdown("### Preview Hasil Transformasi - studentAssessment_agg"))
display(studentAssessment_agg.head(10))


# Cek hasil integrasi
integrasi_summary = pd.DataFrame([
    ["studentInfo_clean", studentInfo_clean.shape[0], studentInfo_clean.shape[1]],
    ["studentRegistration_clean", studentRegistration_clean.shape[0], studentRegistration_clean.shape[1]],
    ["studentVle_agg", studentVle_agg.shape[0], studentVle_agg.shape[1]],
    ["studentAssessment_agg", studentAssessment_agg.shape[0], studentAssessment_agg.shape[1]],
    ["courses_clean", courses_clean.shape[0], courses_clean.shape[1]],
    ["merged_data", merged_data.shape[0], merged_data.shape[1]]
], columns=["Dataset", "Jumlah Baris", "Jumlah Kolom"])

display(integrasi_summary)

,Dataset Hasil,Baris Sebelum,Baris Sesudah,Kolom/Fitur Baru
0,studentVle_agg,10655280,29228,"id_student, code_module, code_presentation, to..."
1,studentAssessment_agg,173912,25843,"id_student, code_module, code_presentation, av..."


### Preview Hasil Transformasi - studentVle_agg

,id_student,code_module,code_presentation,total_clicks,avg_clicks,last_interaction_day,active_days
0,6516,AAA,2014J,2791,4.216012,269,159
1,8462,DDD,2013J,646,2.153333,118,56
2,8462,DDD,2014J,10,2.500000,10,1
3,11391,AAA,2013J,934,4.765306,253,40
4,23629,BBB,2013B,161,2.728814,87,16
5,23698,CCC,2014J,910,2.983607,231,70
6,23798,BBB,2013J,590,1.928105,267,77
7,24186,GGG,2014B,184,2.243902,240,23
8,24213,DDD,2014B,1992,2.695535,237,118
9,24391,GGG,2013J,712,3.095652,249,58


### Preview Hasil Transformasi - studentAssessment_agg

,id_student,code_module,code_presentation,avg_score,max_score,total_assessments,total_submitted,assessment_completion_rate
0,6516,AAA,2014J,61.800000,77.0,5,5,1.0
1,8462,DDD,2013J,87.666667,93.0,3,3,1.0
2,8462,DDD,2014J,86.500000,93.0,4,0,0.0
3,11391,AAA,2013J,82.000000,85.0,5,5,1.0
4,23629,BBB,2013B,82.500000,100.0,4,4,1.0
5,23698,CCC,2014J,74.444444,94.0,9,9,1.0
6,23798,BBB,2013J,93.909091,100.0,11,11,1.0
7,24186,GGG,2014B,62.500000,80.0,8,8,1.0
8,24213,DDD,2014B,76.285714,91.0,7,7,1.0
9,24391,GGG,2013J,88.888889,100.0,9,9,1.0


,Dataset,Jumlah Baris,Jumlah Kolom
0,studentInfo_clean,32593,12
1,studentRegistration_clean,32593,6
2,studentVle_agg,29228,7
3,studentAssessment_agg,25843,8
4,courses_clean,22,3
5,merged_data,32593,25


**KOLOM/VARIABEL BARU UNTUK SISTEM EARLY WARNING**

In [ ]:
# Encode target sederhana: fail/withdrawal = berisiko
merged_data["risk_label"] = merged_data["final_result"].apply(
    lambda x: 1 if x in ["Fail", "Withdrawn", "Withdrawal"] else 0
)

# Low engagement flag
merged_data["low_engagement"] = (merged_data["total_clicks"] < merged_data["total_clicks"].median()).astype(int)

# Low score flag
merged_data["low_score"] = (merged_data["avg_score"] < merged_data["avg_score"].median()).astype(int)

# Simpan hasil processing
processed_path = "/content/drive/MyDrive/Tugas Big-Data/OULAD_EarlyWarning/data/processed/"
merged_data.to_csv(processed_path + "merged_processed_oulad.csv", index=False)

print("Data processing selesai.")
print("Ukuran final data:", merged_data.shape)
display(merged_data.head())

Data processing selesai.
Ukuran final data: (32593, 28)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,active_days,avg_score,max_score,total_assessments,total_submitted,assessment_completion_rate,module_presentation_length,risk_label,low_engagement,low_score
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,40.0,82.0,85.0,5.0,5.0,1.0,268,0,0,0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,80.0,66.4,70.0,5.0,5.0,1.0,268,0,0,1
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,12.0,0.0,0.0,0.0,0.0,0.0,268,1,1,1
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,123.0,76.0,88.0,5.0,5.0,1.0,268,0,0,0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,70.0,54.4,75.0,5.0,5.0,1.0,268,0,0,1
